# Bronze → Silver: Clean Raw Signals

This notebook reads raw signal data from the **bronze** layer (`bronze.raw_lifetime`, `bronze.raw_piece_info`), applies all cleaning rules, and writes validated pieces to the **silver** layer (`silver.clean_pieces`).

**Incremental**: only processes rows with timestamps newer than the latest entry in silver.

### All cleaning happens here

Silver must contain **only valid pieces**. The cleaning rules are:

1. Drop the upsetting signal (incorrectly calculated at the PLC)
2. Remove zero values (tracking failures)
3. Deduplicate timestamps per signal
4. Pivot lifetime signals into columns and join with piece identification
5. Drop rows missing piece_id or die_matrix
6. Remove extreme outliers (Q3 + 3×IQR per signal per die matrix)
7. Validate monotonic cumulative order: 2nd strike < 3rd strike < 4th strike < auxiliary press < bath
8. Compute OEE cycle time (rolling average of last 5 inter-piece intervals) and filter to 11–16s

See [03_CleaningUpData.md](../docs/03_CleaningUpData.md) for the full rationale behind each rule.

In [ ]:
# TODO: implement this cell


## Step 1: Determine incremental boundary

Find the latest timestamp already processed in silver. Only bronze rows after this point will be processed.

In [ ]:
# TODO: implement this cell


## Step 2: Extract and filter raw signals

Read from `bronze.raw_lifetime` excluding:
- The **upsetting signal** — incorrectly calculated at the PLC, values are meaningless (range 0–6.7s with 22.8% zeros)
- **Zero values** — tracking failures where the PLC did not detect the piece at a stage

In [ ]:
# TODO: implement this cell


In [ ]:
# TODO: implement this cell


## Step 3: Deduplicate timestamps

The PLC occasionally double-registers the same piece at the same timestamp. Keep only the last reading per signal.

In [ ]:
# TODO: implement this cell


## Step 4: Pivot and join

Transform the tall signal/value format into one row per piece with all cumulative times as columns. Join lifetime data with piece identification on timestamp.

In [ ]:
# TODO: implement this cell


## Step 5: Normalize and drop missing identification

Map column names to the silver schema, cast die_matrix to integer, and remove pieces missing piece_id or die_matrix.

In [ ]:
# TODO: implement this cell


## Step 6: Remove extreme outliers per die matrix

Pieces with cumulative times far outside the normal range are not slow pieces — they are **tracking failures**: stuck pieces, unclosed cycles, or machine stops that inflated the timer.

For each lifetime column, compute Q1, Q3, and IQR **per die matrix**, then remove rows where any value falls outside `[Q1 - 3×IQR, Q3 + 3×IQR]`.

In [ ]:
# TODO: implement this cell


## Step 7: Validate monotonic cumulative order

Each piece must have increasing cumulative times along the physical process:

**2nd strike < 3rd strike < 4th strike < auxiliary press < bath**

A violation means the PLC misattributed a reading or a tracking cycle overlapped. These are not valid pieces.

In [ ]:
# TODO: implement this cell


## Step 8: Compute OEE cycle time and filter

The OEE cycle time measures the **time between consecutive pieces** exiting the line — it is a production throughput metric. The original PLC computes it as the rolling average of the last 5 pieces at the auxiliary press.

Since the auxiliary press signal is not in our dataset, we approximate it from the **timestamp intervals** between consecutive pieces, using a rolling window of 5.

Valid OEE cycle time is **11–16 seconds**. Values outside this range indicate production stops, changeovers, or sensor errors. Pieces with invalid OEE are kept in silver (they are valid pieces) but their `oee_cycle_time_s` is set to NULL.

In [ ]:
# TODO: implement this cell


## Step 9: Write to silver

Append the cleaned, validated pieces to `silver.clean_pieces`.

In [ ]:
# TODO: implement this cell


## Cleaning Report

Summary of all cleaning decisions applied in this run, with counts and justifications.

In [ ]:
# TODO: implement this cell
